# Detecção e diagnóstico de falhas em rolamentos — dados MFPT

ECM514 — Ciência de Dados, 2º semestre de 2026.

Máquinas que giram vibram de um jeito quando o rolamento está bom e de outro quando ele tem um defeito. Usamos gravações reais de vibração para responder duas perguntas: o rolamento tem algum problema (detecção) e, se tem, qual é o defeito (diagnóstico).

## 1. Os dados

Os dados da MFPT são gravações da vibração de um rolamento em bancada, em três situações: rolamento normal, defeito na pista externa e defeito na pista interna. Cada arquivo traz o sinal de vibração (em g), quantas medições foram feitas por segundo, a carga sobre o rolamento e a velocidade do eixo (25 voltas por segundo em todas).

Ao baixar o dataset, ele vem com uma pasta de análises, que só repete um dos arquivos de pista interna junto com os scripts de MATLAB da MFPT, e com uma pasta de exemplos de máquinas reais (uma bomba de óleo, um rolamento de velocidade intermediária e um rolamento planetário). Desconsideramos as duas: a cópia contaria a mesma gravação duas vezes, e os exemplos reais são outras máquinas, gravadas de outro jeito e sem a indicação de normal, pista externa ou pista interna. Ficamos com as 20 gravações de bancada.

As gravações normais foram feitas com 97.656 medições por segundo, e as de pista interna com 48.828. Se deixássemos assim, o modelo poderia separar as classes por esse detalhe da gravação, e não pelo defeito. Como 97.656 é exatamente o dobro de 48.828, passamos todas para 48.828: o sinal é filtrado e fica uma medição a cada duas. Isso corta a vibração acima de 24 mil ciclos por segundo, que as gravações de 48.828 nunca tiveram; as faixas que a própria MFPT usa nas análises dela ficam abaixo disso.

Os sinais também não oscilam exatamente em torno de zero, e esse deslocamento muda de uma sessão de gravação para outra. Como ele é do sensor e não do rolamento, tiramos a média de cada gravação.

In [ ]:
# Baixa a versão 1 do dataset do Kaggle (o kagglehub devolve a pasta onde os arquivos ficaram) e
# mostra as versões das bibliotecas, para que o resultado possa ser reproduzido.
import os
import sys
import glob
import hashlib
import numpy as np
import pandas as pd
import scipy
import sklearn
import matplotlib
import matplotlib.pyplot as plt
import kagglehub
from scipy.io import loadmat
from scipy.signal import decimate

path = kagglehub.dataset_download('emperorpein/mfpt-fault-datasets/versions/1')
print(f'Python {sys.version.split()[0]} | numpy {np.__version__} | pandas {pd.__version__} | '
      f'scipy {scipy.__version__} | scikit-learn {sklearn.__version__} | matplotlib {matplotlib.__version__}')

In [ ]:
# Lê só as 20 gravações de bancada (sem a pasta de análises e sem as máquinas reais) e confere a
# impressão digital (SHA-256) de cada arquivo, para garantir que são exatamente os dados analisados.
# Monta uma tabela com uma linha por gravação; a classe sai do nome do arquivo, e a média de cada
# sinal é tirada porque ela vem do sensor, não do rolamento.
sha256_esperado = {
    'baseline_1': '540e5f4fcc127d3ddb65c42465cdf7d29696e2906ff8fe52ef980470469d066a',
    'baseline_2': '4374ae7f04607474e55b00d19a2ed48bdaa46d699624ff34d4a46ba4c7ca478b',
    'baseline_3': 'f03f63313760ef5f01b977db6b1fee12b3ba76a2f32562f3d4fcfc75e963b4a9',
    'OuterRaceFault_1': 'c3907e7ecb5cb98f37d5f5faebfd8ed3467875d83a3c6505c6e4a365272cdc83',
    'OuterRaceFault_2': '40d11431b262d75062fa66574ef1e441897df359452324964ea62697e43a70bc',
    'OuterRaceFault_3': '16d3c33ae3135281705877ca4aa824cf10d7705104ec4a626385aef5ecdce194',
    'OuterRaceFault_vload_1': '5651c292924ff97b1384d7a81006d4a224a052c4b97a4524c5d5ffdf6a9ce1c9',
    'OuterRaceFault_vload_2': '348a4e2750d339f6f9101f5c74ea631486cc11098d9f282d30840dfb0e59f4cd',
    'OuterRaceFault_vload_3': '0fbc3b01ceb373cc656ce6fc7cc945df0a2f6afbae1f3a2efa9c598020cd0685',
    'OuterRaceFault_vload_4': 'e814bbb6e5741e1254595819a70d7f62d1f9a795588e452eeb9bfc8e50d15e61',
    'OuterRaceFault_vload_5': '93531915d0d2c8d6a9933d2541f43b3eb48a20fe9fecbf5a872f68566f5a53ad',
    'OuterRaceFault_vload_6': '4126cff4a1c21d253741144e9e608f8bb3475e7343676c60fef2249179569fd9',
    'OuterRaceFault_vload_7': '82ab4d7df2785e430a817180a541e741bc0ab633d7e2434dc363620ea1b1a5ee',
    'InnerRaceFault_vload_1': '2fd8456317ba45bb054f622b8cb4a401f937d6e830564f85efd41bf81ba8d7aa',
    'InnerRaceFault_vload_2': '08251ffd88c956dcd9f52a04f66d02f6618107110a90ec46d7f114fba2a3277c',
    'InnerRaceFault_vload_3': '5f63f72f6ee173a3a9620db7acb37a9fadae107ffdfd77598ca90d67dd00d066',
    'InnerRaceFault_vload_4': '7c2e753cae90a52de4d430c5fa723ec8a57980cff3521890343b181c2fdfcf75',
    'InnerRaceFault_vload_5': '427053478152e9d259e8c4ae9ec18932ab538fd808a26611e98168fa989e7860',
    'InnerRaceFault_vload_6': '97136f3135115faf25d12719aa57fb22f2bd521e98447b804be0dbe21161168c',
    'InnerRaceFault_vload_7': 'e6545bb0414c58d159ef0188ac258c983cd3696fd44662d2dea9898513309651',
}
classes = {'baseline': 'normal', 'OuterRace': 'pista externa', 'InnerRace': 'pista interna'}

arquivos = sorted(f for f in glob.glob(os.path.join(path, '**', '*.mat'), recursive=True)
                  if '5 - Analyses' not in f and '6 - Real World' not in f)

linhas = []
for f in arquivos:
    nome = os.path.basename(f).replace('.mat', '')
    with open(f, 'rb') as arquivo:
        assert hashlib.sha256(arquivo.read()).hexdigest() == sha256_esperado[nome], f'arquivo diferente: {nome}'
    b = loadmat(f, squeeze_me=True, struct_as_record=False)['bearing']
    classe = next(c for chave, c in classes.items() if nome.startswith(chave))
    sinal = np.asarray(b.gs, dtype=float)
    linhas.append({'arquivo': nome, 'classe': classe, 'medicoes_por_seg': int(b.sr),
                   'carga': int(b.load), 'giro_hz': float(b.rate), 'media_original': sinal.mean(), 'sinal': sinal - sinal.mean()})

gravacoes = pd.DataFrame(linhas)
assert len(gravacoes) == len(sha256_esperado)
gravacoes['duracao_s'] = gravacoes.sinal.apply(len) / gravacoes.medicoes_por_seg
gravacoes.drop(columns='sinal').round(3)

In [ ]:
# Passa as gravações de 97.656 para 48.828 medições por segundo, para que a taxa de gravação não
# denuncie a classe. Depois resume cada classe com a força média da vibração (rms) de cada gravação.
taxa = 48828

gravacoes['sinal'] = [decimate(s, 2) if sr == 2 * taxa else s
                      for s, sr in zip(gravacoes.sinal, gravacoes.medicoes_por_seg)]
gravacoes['medicoes_por_seg'] = taxa
gravacoes['duracao_s'] = gravacoes.sinal.apply(len) / taxa
gravacoes['rms'] = gravacoes.sinal.apply(lambda s: np.sqrt(np.mean(s ** 2)))
gravacoes['pico'] = gravacoes.sinal.apply(lambda s: np.abs(s).max())

print(gravacoes[['arquivo', 'classe', 'carga', 'rms', 'pico']].round(2).to_string(index=False))
gravacoes.groupby('classe').agg(gravacoes=('arquivo', 'count'), segundos=('duracao_s', 'sum'),
                                rms_min=('rms', 'min'), rms_max=('rms', 'max'),
                                pico_min=('pico', 'min'), pico_max=('pico', 'max')).round(2)

In [ ]:
# Mostra 0,1 s de uma gravação de cada classe, na mesma escala, para ver a cara de cada defeito.
# Escolhemos gravações com carga 270 (ou a mais próxima, no caso da pista interna).
exemplos = ['baseline_1', 'OuterRaceFault_1', 'InnerRaceFault_vload_6']
n = int(0.1 * taxa)
t = np.arange(n) / taxa

fig, axes = plt.subplots(3, 1, figsize=(10, 6), sharex=True, sharey=True)
for ax, nome in zip(axes, exemplos):
    linha = gravacoes.set_index('arquivo').loc[nome]
    ax.plot(t, linha.sinal[:n], lw=0.6)
    ax.set_title(f'{linha.classe} ({nome}, carga {linha.carga})', fontsize=10)
    ax.set_ylabel('vibração (g)')
axes[-1].set_xlabel('tempo (s)')
plt.tight_layout()
plt.show()

### Resultados

Ficamos com 78 segundos de sinal, todos com 48.828 medições por segundo: 18 s de rolamento normal (3 gravações), 39 s com defeito na pista externa (10 gravações) e 21 s com defeito na pista interna (7 gravações).

O deslocamento do sensor confirma que ele vem da sessão de gravação, e não do defeito: fica entre -0,06 e -0,14 nas gravações de 97.656, e muda aos poucos, arquivo após arquivo, nas de 48.828 (de -0,19 a -0,16 na pista externa e de -0,23 a -0,19 na pista interna). Se não fosse tirado, o modelo poderia usá-lo para reconhecer a sessão.

A pista interna se separa fácil: força média da vibração (rms) entre 1,73 e 2,07 g e picos de 23 a 46 g, contra 0,74 a 0,75 g e picos de 3,4 a 3,8 g no rolamento normal. No gráfico ela aparece como pancadas fortes e espaçadas, de até 18 g nesse trecho.

A pista externa é o caso difícil. Nas três gravações feitas nas mesmas condições do normal (carga 270, mesma taxa de gravação), a força média fica em 0,55, 0,68 e 0,87 g, uma abaixo e outra acima do normal, e os picos (3,3 a 5,1 g) também se misturam com os do normal. Nem a força média nem o pico, sozinhos, separam essas gravações. As outras sete têm picos maiores (5 a 19 g), mas não existe gravação normal com essas cargas para comparar.

Limites dos dados:
- só há 3 gravações normais, todas com carga 270, gravadas nas mesmas condições das três primeiras de pista externa;
- a pista interna nunca foi gravada nas mesmas condições do normal, então medidas genéricas do sinal (força média, pico) podem estar captando a sessão de gravação junto com o defeito;
- a comparação mais justa que os dados permitem é normal contra as três gravações de pista externa com carga 270.

## 2. O ritmo de cada defeito

Um defeito no rolamento causa uma pancada cada vez que uma esfera passa por ele. Com as medidas do rolamento que vêm no próprio dataset (8 esferas, diâmetro da esfera 0,235 e diâmetro do círculo das esferas 1,245) e a velocidade do eixo, dá para calcular quantas pancadas por segundo cada defeito produz (BECHHOEFER; MENON, 2009). Para enxergar esse ritmo, ficamos com a faixa de 2 a 4 mil ciclos por segundo do sinal, tiramos o contorno da vibração (a envoltória) e medimos quanto ele se repete em cada frequência, comparado com o fundo (KIM; AN; CHOI, 2020). A faixa de 2 a 4 mil ciclos por segundo é a que a MFPT usa no script de análise distribuído com os dados; ela não foi otimizada para este trabalho. Esse ritmo vem da geometria do rolamento e não da sessão de gravação, por isso ele é a evidência mais segura de que existe um defeito.

In [ ]:
# Calcula o ritmo de pancadas de cada defeito pela geometria do rolamento e mede, em cada gravação,
# quanto o contorno da vibração se repete nesse ritmo, comparado com o fundo (de 50 a 200 Hz).
# Também mostra em que frequência fica o maior pico do contorno entre 50 e 200 Hz.
from scipy.signal import butter, sosfiltfilt, hilbert, welch

esfera, circulo, n_esferas = 0.235, 1.245, 8
razao = esfera / circulo
ritmo_externa = 0.5 * (1 - razao) * n_esferas
ritmo_interna = 0.5 * (1 + razao) * n_esferas

faixa = butter(4, [2000, 4000], btype='band', fs=taxa, output='sos')

def espectro_envoltoria(sinal):
    envoltoria = np.abs(hilbert(sosfiltfilt(faixa, sinal)))
    return welch(envoltoria - envoltoria.mean(), taxa, nperseg=2 ** 16)

def destaque(freq, potencia, alvo):
    perto = (freq > 0.97 * alvo) & (freq < 1.03 * alvo)
    fundo = np.median(potencia[(freq > 50) & (freq < 200)])
    return potencia[perto].max() / fundo

def frequencia_do_pico(freq, potencia):
    faixa_busca = (freq > 50) & (freq < 200)
    return freq[faixa_busca][np.argmax(potencia[faixa_busca])]

espectros = gravacoes.sinal.apply(espectro_envoltoria)
gravacoes['externa_x_fundo'] = [destaque(f, p, ritmo_externa * g)
                                for (f, p), g in zip(espectros, gravacoes.giro_hz)]
gravacoes['interna_x_fundo'] = [destaque(f, p, ritmo_interna * g)
                                for (f, p), g in zip(espectros, gravacoes.giro_hz)]
gravacoes['pico_hz'] = [frequencia_do_pico(f, p) for f, p in espectros]

print(f'pista externa: {ritmo_externa * 25:.1f} pancadas por segundo | '
      f'pista interna: {ritmo_interna * 25:.1f} pancadas por segundo (eixo a 25 voltas por segundo)')
gravacoes[['arquivo', 'classe', 'pico_hz', 'externa_x_fundo', 'interna_x_fundo']].round(1)

In [ ]:
# Mostra o espectro do contorno das mesmas três gravações, com as linhas no ritmo esperado de cada
# defeito, e o maior valor de cada espectro até 300 Hz, já que os gráficos têm escalas diferentes.
fig, axes = plt.subplots(3, 1, figsize=(10, 6), sharex=True)
for ax, nome in zip(axes, exemplos):
    i = gravacoes.index[gravacoes.arquivo == nome][0]
    freq, potencia = espectros[i]
    ax.plot(freq, potencia, lw=0.8)
    ax.axvline(ritmo_externa * gravacoes.giro_hz[i], color='tab:orange', ls='--', label='ritmo da pista externa')
    ax.axvline(ritmo_interna * gravacoes.giro_hz[i], color='tab:red', ls='--', label='ritmo da pista interna')
    ax.set_title(f'{gravacoes.classe[i]} ({nome})', fontsize=10)
    ax.set_ylabel('potência')
    print(f'{nome}: maior valor até 300 Hz = {potencia[freq < 300].max():.2e}')
axes[0].legend(fontsize=8)
axes[-1].set_xlim(0, 300)
axes[-1].set_xlabel('frequência (pancadas por segundo)')
plt.tight_layout()
plt.show()

### Resultados

Pela geometria do rolamento, com o eixo a 25 voltas por segundo, o defeito na pista externa deveria causar 81,1 pancadas por segundo, e o da pista interna 118,9. É o que aparece. Nas 10 gravações de pista externa, o contorno da vibração se repete no ritmo da pista externa de 422,8 a 2.380,5 vezes mais do que o fundo. Nas 7 de pista interna, se repete no ritmo da pista interna de 1.301,0 a 1.996,9 vezes mais do que o fundo. Nas 3 normais, os dois ritmos ficam só de 1,6 a 4,1 vezes o fundo, ou seja, não aparecem. O maior pico do contorno entre 50 e 200 Hz cai em 80,5 Hz em 9 das 10 gravações de pista externa (79,7 Hz na outra) e em 117,7 Hz em todas as de pista interna, menos de 2% abaixo da conta, o que é compatível com o eixo girando um pouco abaixo de 25 voltas por segundo.

No gráfico do contorno, a pista externa mostra o pico no seu ritmo e um menor no dobro dele. A pista interna mostra o pico no seu ritmo e vários outros espaçados de 25 em 25, a velocidade do eixo, padrão descrito na literatura como típico de defeito na pista interna (KIM; AN; CHOI, 2020). Os três gráficos têm escalas diferentes: o maior valor até 300 Hz é 2,44e-04 na gravação normal e 1,48e-01 na de pista interna, cerca de 600 vezes maior.

A sessão de gravação pode mudar a força média ou o deslocamento do sensor, mas não cria uma pancada que se repete exatamente no ritmo previsto pela geometria do rolamento. Isso vale também para as três gravações de pista externa feitas nas mesmas condições do normal, em que nem a força média nem o pico separavam as classes.

## 3. Janelas e medidas

Para treinar e avaliar os modelos, cortamos cada gravação em janelas de 0,25 s e calculamos dois grupos de medidas para cada janela. As medidas genéricas olham o tamanho e o formato das pancadas: força média (rms), maior pico, curtose (o quanto o sinal é feito de pancadas isoladas) e pico dividido pela força média. As medidas do ritmo olham se o contorno da vibração se repete no ritmo de cada defeito.

Janelas vizinhas da mesma gravação são quase iguais: mesmo rolamento, mesma carga, mesma montagem. Se uma janela fica no treino e a vizinha na avaliação, o modelo pode acertar só por reconhecer a gravação (ROBERTS et al., 2017; KAPOOR; NARAYANAN, 2023). Por isso toda avaliação deste trabalho é feita por gravação: uma gravação usada para avaliar nunca tem janelas no treino. As seções 4 e 5 descrevem como isso é feito em cada etapa.

In [ ]:
# Corta cada gravação em janelas de 0,25 s e calcula as medidas de cada janela. As genéricas
# (força média, pico, curtose e pico dividido pela força média) olham só o tamanho e o formato das
# pancadas; as do ritmo olham se o contorno se repete no ritmo de cada defeito.
from scipy.stats import kurtosis

tamanho = int(0.25 * taxa)
genericas = ['rms', 'pico', 'curtose', 'pico_por_rms']
ritmo = ['externa_x_fundo', 'interna_x_fundo']

linhas = []
for _, g in gravacoes.iterrows():
    envoltoria = np.abs(hilbert(sosfiltfilt(faixa, g.sinal)))
    for k in range(len(g.sinal) // tamanho):
        trecho = slice(k * tamanho, (k + 1) * tamanho)
        s, e = g.sinal[trecho], envoltoria[trecho]
        freq, potencia = welch(e - e.mean(), taxa, nperseg=tamanho)
        rms = np.sqrt(np.mean(s ** 2))
        linhas.append({'arquivo': g.arquivo, 'classe': g.classe, 'carga': g.carga,
                       'rms': rms, 'pico': np.abs(s).max(), 'curtose': kurtosis(s),
                       'pico_por_rms': np.abs(s).max() / rms,
                       'externa_x_fundo': destaque(freq, potencia, ritmo_externa * g.giro_hz),
                       'interna_x_fundo': destaque(freq, potencia, ritmo_interna * g.giro_hz)})

janelas = pd.DataFrame(linhas)
print(janelas.groupby('classe').agg(gravacoes=('arquivo', 'nunique'), janelas=('arquivo', 'size')))
print()
print('ritmo do próprio defeito nas janelas (vezes o fundo):')
print(pd.DataFrame({
    'normal, ritmo externo': janelas.loc[janelas.classe == 'normal', 'externa_x_fundo'].describe(),
    'normal, ritmo interno': janelas.loc[janelas.classe == 'normal', 'interna_x_fundo'].describe(),
    'pista externa, ritmo externo': janelas.loc[janelas.classe == 'pista externa', 'externa_x_fundo'].describe(),
    'pista interna, ritmo interno': janelas.loc[janelas.classe == 'pista interna', 'interna_x_fundo'].describe(),
}).round(1).loc[['min', '50%', 'max']])

### Resultados

Foram 312 janelas: 72 normais (3 gravações), 156 de pista externa (10 gravações) e 84 de pista interna (7 gravações).

Numa janela de 0,25 s o ritmo do defeito aparece bem mais fraco do que na gravação inteira, porque a janela curta tem menos pancadas para medir. Nas janelas de pista externa, o ritmo da pista externa fica entre 27,8 e 403,6 vezes o fundo (mediana de 116,5). Nas de pista interna, o ritmo da pista interna fica entre 9,0 e 81,0 vezes o fundo (mediana de 20,5). Nas janelas normais, os dois ritmos chegam no máximo a 5,3 e 11,6 vezes o fundo. Janelas mais longas deixariam o ritmo mais claro, mas dariam menos janelas normais para treinar.

## 4. Detecção

Na detecção, o modelo aprende apenas com janelas normais e não sabe que tipos de defeito existem; ele só aprende como é o normal. Usamos duas técnicas. O Isolation Forest faz cortes ao acaso nos dados: uma janela que fica isolada com poucos cortes é diferente das demais (LIU; TING; ZHOU, 2008). A distância com PCA mede quantos desvios-padrão do normal uma janela está, nas direções principais dos dados normais, como no controle estatístico de processos baseado em PCA (PÉREZ-TORRES et al., 2026). Cada técnica roda duas vezes, uma com cada grupo de medidas.

A avaliação deixa uma gravação normal inteira de fora a cada rodada. Como há 3 gravações normais, são 3 rodadas: o detector aprende com as janelas das outras duas, o alarme falso (FPR) é medido nas janelas da gravação normal que ficou de fora, e a detecção (TPR) em todas as janelas com defeito. O detector nunca viu nenhuma das janelas avaliadas. O limite de alerta é a janela normal mais estranha do treino da rodada: o que passar dela é anomalia. Esse limite foi escolhido antes de olhar os resultados, para não ajustá-lo aos dados avaliados.

In [ ]:
# Detecção avaliada deixando uma gravação normal inteira de fora a cada rodada (3 rodadas). O detector
# aprende só com as janelas das outras duas gravações normais e não sabe que defeitos existem; o
# limite de alerta é a janela normal mais estranha desse treino. O alarme falso (FPR) é medido na
# gravação normal que ficou de fora e a detecção (TPR) em todas as janelas com defeito: o detector
# nunca viu nenhuma delas. O Isolation Forest isola as janelas com cortes ao acaso (o que fica
# isolado rápido é estranho); a distância com PCA mede quantos desvios-padrão do normal a janela está.
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.ensemble import IsolationForest

def estranheza_isolation(treino, avaliado):
    modelo = IsolationForest(random_state=42).fit(treino)
    return -modelo.score_samples(treino), -modelo.score_samples(avaliado)

def estranheza_pca(treino, avaliado):
    scaler = StandardScaler().fit(treino)
    pca = PCA(whiten=True).fit(scaler.transform(treino))
    distancia = lambda X: np.linalg.norm(pca.transform(scaler.transform(X)), axis=1)
    return distancia(treino), distancia(avaliado)

janelas['grupo'] = np.where((janelas.classe == 'pista externa') & (janelas.carga == 270),
                            'pista externa, carga 270', janelas.classe)
normais = sorted(gravacoes.arquivo[gravacoes.classe == 'normal'])
tecnicas = {'Isolation Forest': estranheza_isolation, 'Distância com PCA': estranheza_pca}
grupos_medidas = {'genéricas': genericas, 'ritmo do defeito': ritmo}

rodadas_deteccao, alertas = [], []
for tecnica, estranheza in tecnicas.items():
    for nome_medidas, colunas in grupos_medidas.items():
        for fora in normais:
            treino_normal = janelas[(janelas.classe == 'normal') & (janelas.arquivo != fora)]
            avaliado = janelas[(janelas.arquivo == fora) | (janelas.classe != 'normal')].copy()
            nota_treino, nota = estranheza(treino_normal[colunas], avaliado[colunas])
            avaliado['alerta'] = nota > nota_treino.max()
            avaliado['tecnica'], avaliado['medidas'], avaliado['normal_fora'] = tecnica, nome_medidas, fora
            defeito = avaliado.classe != 'normal'
            rodadas_deteccao.append({'técnica': tecnica, 'medidas': nome_medidas, 'normal deixada de fora': fora,
                                     'TPR': avaliado.alerta[defeito].mean(), 'FPR': avaliado.alerta[~defeito].mean()})
            alertas.append(avaliado)

rodadas_deteccao = pd.DataFrame(rodadas_deteccao)
alertas = pd.concat(alertas)
print(rodadas_deteccao.round(3).to_string(index=False))
deteccao = rodadas_deteccao.groupby(['técnica', 'medidas'], sort=False)[['TPR', 'FPR']].mean()
deteccao.round(3)

In [ ]:
# Quanto cada combinação detecta em cada grupo de gravações, na média das 3 rodadas. O grupo que
# importa é a pista externa com carga 270, gravada nas mesmas condições do normal. Para a distância
# com PCA com as medidas do ritmo, lista as janelas com defeito que não geraram alerta.
print(alertas.groupby(['tecnica', 'medidas', 'grupo'], sort=False).alerta.mean().unstack().round(3).to_string())
print()
perdidas = alertas[(alertas.tecnica == 'Distância com PCA') & (alertas.medidas == 'ritmo do defeito')
                   & (alertas.classe != 'normal') & ~alertas.alerta]
print('janelas com defeito sem alerta (distância com PCA, ritmo do defeito):')
print(perdidas[['normal_fora', 'arquivo', 'carga', 'externa_x_fundo', 'interna_x_fundo']].round(1).to_string())
print()
print('maior ritmo interno numa janela normal:', janelas.loc[janelas.classe == 'normal', 'interna_x_fundo'].max().round(1))

### Resultados

Média das 3 rodadas:

| Técnica | Medidas | Detecta defeito (TPR) | Alarme falso (FPR) | Pista externa, carga 270 | Pista interna |
|---|---|---|---|---|---|
| Isolation Forest | genéricas | 0,889 | 0,014 | 0,630 | 1,000 |
| Isolation Forest | ritmo do defeito | 0,107 | 0,000 | 0,005 | 0,302 |
| Distância com PCA | genéricas | 1,000 | 0,014 | 1,000 | 1,000 |
| Distância com PCA | ritmo do defeito | 0,990 | 0,028 | 1,000 | 0,972 |

A distância com PCA funcionou com os dois grupos de medidas. Com as medidas do ritmo, detectou 99,0% das janelas com defeito, todas as de pista externa. As únicas janelas que ela deixou passar são da gravação de pista interna com carga 200, em que o ritmo da pista interna ficou entre 9,0 e 11,1 vezes o fundo, abaixo da janela normal com o ritmo mais alto por acaso (11,6). O alarme falso dela foi de 0 nas rodadas sem a baseline_1 e sem a baseline_3, e de 8,3% (2 de 24 janelas) na rodada sem a baseline_2. Com as medidas genéricas, detectou todas as janelas com defeito, com 1 alarme falso (4,2% das janelas da baseline_1) e nenhum nas outras rodadas. Combinando as quatro medidas genéricas, a distância com PCA separou do normal também as janelas de pista externa com carga 270, embora a força média e o pico sozinhos não separem essas gravações.

O Isolation Forest falhou com as medidas do ritmo, detectando só 10,7% dos defeitos. O motivo está em como ele funciona: os cortes são feitos entre o menor e o maior valor de cada medida no treino (LIU; TING; ZHOU, 2008), então uma janela com ritmo muito acima do normal recebe quase a mesma nota de uma janela normal que, por acaso, ficou no ponto mais alto do treino. Como o limite de alerta é justamente a janela normal mais estranha, os defeitos quase nunca passam dele. Com as medidas genéricas ele foi melhor (88,9%, com 1,4% de alarme falso), mas detectou só 63,0% das janelas de pista externa com carga 270, o caso mais difícil.

Cada rodada mede o alarme falso numa única gravação normal (24 janelas). Com só 3 gravações normais, essas taxas são pouco precisas: uma janela a mais com alerta muda o alarme falso da rodada em 4,2 pontos percentuais.

## 5. Diagnóstico

No diagnóstico, o modelo aprende com as três classes e precisa dizer qual é o defeito de cada janela. Usamos os seguintes modelos: regressão logística, Knn, árvore de decisão e Random Forest, cada um com os dois grupos de medidas, (SPERANDEI, 2014; COVER; HART, 1967; QUINLAN, 1986; BREIMAN, 2001), todos com a implementação do scikit-learn (PEDREGOSA et al., 2011).

Primeiro comparamos duas formas de validação cruzada. Na comum, as janelas são sorteadas entre as rodadas, então quase sempre uma janela cai no treino e a vizinha na avaliação. Na separada por gravação, cada gravação fica inteira de um lado só. São 3 rodadas, e cada uma deixa de fora uma das 3 gravações normais e cerca de um terço das gravações de cada defeito. A diferença entre as duas mostra o quanto o modelo acerta só por reconhecer a gravação.

A avaliação final também é por gravação. Em cada uma das 3 rodadas, as gravações da rodada ficam de fora; com as demais, o melhor modelo é escolhido por validação separada por gravação, treinado e usado para classificar as gravações de fora. Assim a escolha do modelo nunca vê as gravações em que ele é avaliado. Juntando as 3 rodadas, cada janela é classificada uma vez, por um modelo que não viu a gravação dela, e daí saem a matriz de confusão e o acerto por classe.

Também testamos o PCA para resumir as medidas em poucos componentes antes do classificador. Juntamos as 6 medidas, padronizamos e ficamos com os componentes que somam 70% da variação, um corte comum na prática (JOLLIFFE; CADIMA, 2016), e, para comparar, 90%. Comparamos com as 6 medidas juntas sem PCA, pela validação separada por gravação.

In [ ]:
# Compara a validação cruzada comum (janelas sorteadas) com a separada por gravação, para cada
# modelo e grupo de medidas. A diferença entre as duas mostra o quanto o modelo acerta só por
# reconhecer a gravação. As 3 rodadas da separada por gravação são montadas no código: dentro de cada
# classe, as gravações são distribuídas em ordem entre as rodadas, então cada rodada deixa de fora
# uma gravação normal e cerca de um terço das gravações de cada defeito.
from sklearn.base import clone
from sklearn.model_selection import cross_val_score, KFold
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import confusion_matrix, classification_report, ConfusionMatrixDisplay

rodada_da_gravacao = {}
for classe, arquivos_classe in gravacoes.groupby('classe').arquivo:
    for k, arquivo in enumerate(sorted(arquivos_classe)):
        rodada_da_gravacao[arquivo] = k % 3
rodada = janelas.arquivo.map(rodada_da_gravacao)
rodadas_por_gravacao = [(np.where(rodada != r)[0], np.where(rodada == r)[0]) for r in range(3)]

models = {'Regressão logística': make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)),
          'Knn (k=3)': make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=3)),
          'Árvore de decisão': DecisionTreeClassifier(random_state=42),
          'Random Forest': RandomForestClassifier(random_state=42)}

cv_results = []
for nome_medidas, colunas in grupos_medidas.items():
    for nome_modelo, model in models.items():
        comum = cross_val_score(model, janelas[colunas], janelas.classe,
                                cv=KFold(5, shuffle=True, random_state=42), scoring='accuracy').mean()
        por_gravacao = cross_val_score(model, janelas[colunas], janelas.classe, cv=rodadas_por_gravacao,
                                       scoring='accuracy').mean()
        cv_results.append({'medidas': nome_medidas, 'modelo': nome_modelo,
                           'validação comum': comum, 'separando por gravação': por_gravacao,
                           'diferença': por_gravacao - comum})

cv_results = pd.DataFrame(cv_results)
cv_results.round(3)

In [ ]:
# Avaliação final do diagnóstico, também por gravação: em cada uma das 3 rodadas, as gravações da
# rodada ficam de fora; com as outras, o melhor modelo é escolhido por validação separada por
# gravação (2 rodadas internas), treinado e usado para classificar as gravações de fora. Assim a
# escolha do modelo nunca vê as gravações em que ele é avaliado. Juntando as 3 rodadas, cada janela
# é classificada uma vez, por um modelo que não viu a gravação dela.
def diagnostico_por_gravacao(colunas):
    previsto = pd.Series(index=janelas.index, dtype=object)
    escolhidos = []
    for r in range(3):
        fora = (rodada == r).to_numpy()
        treino, rodada_treino = janelas[~fora], rodada[~fora]
        internas = [(np.where(rodada_treino != q)[0], np.where(rodada_treino == q)[0])
                    for q in sorted(rodada_treino.unique())]
        notas = {nome: cross_val_score(model, treino[colunas], treino.classe, cv=internas,
                                       scoring='accuracy').mean() for nome, model in models.items()}
        melhor = max(notas, key=notas.get)
        model = clone(models[melhor]).fit(treino[colunas], treino.classe)
        previsto[fora] = model.predict(janelas.loc[fora, colunas])
        escolhidos.append(melhor)
    return previsto, escolhidos

finais = {}
for nome_medidas, colunas in grupos_medidas.items():
    previsto, escolhidos = diagnostico_por_gravacao(colunas)
    finais[nome_medidas] = (previsto, escolhidos)
    certo = previsto == janelas.classe
    print(f'== medidas {nome_medidas}: modelo escolhido em cada rodada: {escolhidos} ==')
    print(classification_report(janelas.classe, previsto, digits=3))
    print('acerto por classe:', certo.groupby(janelas.classe).mean().round(3).to_dict())
    print(f'acerto na pista externa com carga 270: {certo[janelas.grupo == "pista externa, carga 270"].mean():.3f}')
    erros = janelas[~certo].assign(previsto=previsto[~certo])
    print('janelas erradas por gravação:', erros.groupby(['arquivo', 'previsto']).size().to_dict())
    print()

In [ ]:
# Matriz de confusão de cada grupo de medidas, com todas as classes e todas as janelas.
classes_ordem = ['normal', 'pista externa', 'pista interna']
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
for ax, (nome_medidas, (previsto, escolhidos)) in zip(axes, finais.items()):
    cm = confusion_matrix(janelas.classe, previsto, labels=classes_ordem)
    ConfusionMatrixDisplay(cm, display_labels=classes_ordem).plot(ax=ax, colorbar=False)
    ax.set_title(f'medidas {nome_medidas}', fontsize=10)
    ax.set_xlabel('classe prevista')
    ax.set_ylabel('classe real')
plt.tight_layout()
plt.show()

In [ ]:
# Testa o PCA antes do classificador: as 6 medidas juntas, padronizadas e resumidas nos componentes
# que somam 70% ou 90% da variação. O PCA é ajustado só com o treino de cada rodada, por isso fica
# dentro do pipeline. Compara com as 6 medidas juntas sem PCA, pela validação separada por gravação.
todas = genericas + ritmo
pca_results = []
for nome_entrada, variacao in [('6 medidas, sem PCA', None), ('PCA, 70% da variação', 0.7), ('PCA, 90% da variação', 0.9)]:
    for nome_modelo, model in models.items():
        if variacao:
            model = make_pipeline(StandardScaler(), PCA(variacao), clone(model))
        acerto = cross_val_score(model, janelas[todas], janelas.classe, cv=rodadas_por_gravacao,
                                 scoring='accuracy').mean()
        pca_results.append({'entrada': nome_entrada, 'modelo': nome_modelo, 'separando por gravação': acerto})

pca = PCA().fit(StandardScaler().fit_transform(janelas[todas]))
print('variação explicada por componente:', pca.explained_variance_ratio_.round(3))
pd.DataFrame(pca_results).pivot(index='entrada', columns='modelo', values='separando por gravação').round(3)

### Resultados

| Medidas | Modelo | Validação comum | Separando por gravação | Diferença |
|---|---|---|---|---|
| genéricas | Regressão logística | 0,958 | 0,909 | -0,049 |
| genéricas | Knn (k=3) | 0,997 | 0,892 | -0,105 |
| genéricas | Árvore de decisão | 1,000 | 0,900 | -0,100 |
| genéricas | Random Forest | 1,000 | 0,914 | -0,086 |
| ritmo do defeito | Regressão logística | 0,990 | 0,987 | -0,003 |
| ritmo do defeito | Knn (k=3) | 0,987 | 0,990 | 0,002 |
| ritmo do defeito | Árvore de decisão | 0,984 | 0,990 | 0,006 |
| ritmo do defeito | Random Forest | 0,984 | 0,986 | 0,002 |

Com as medidas genéricas, a validação comum promete de 95,8% a 100% de acerto, mas, quando o modelo é avaliado em gravações que nunca viu, o acerto cai para 89,2% a 91,4%. Parte do acerto vinha de reconhecer a gravação, e não o defeito. Com as medidas do ritmo, as duas validações dão praticamente o mesmo resultado (de 98,4% a 99,0%), porque o ritmo do defeito não depende da gravação.

Na avaliação final por gravação:

| Medidas | Modelo escolhido nas 3 rodadas | Acerto geral | Normal | Pista externa | Pista interna | Pista externa, carga 270 |
|---|---|---|---|---|---|---|
| genéricas | regressão logística, Random Forest, regressão logística | 0,910 | 0,917 | 0,859 | 1,000 | 0,806 |
| ritmo do defeito | Knn (k=3) nas três | 0,990 | 0,972 | 1,000 | 0,988 | 1,000 |

Com as medidas genéricas, os erros se concentram onde o normal e a pista externa se parecem: 14 janelas da OuterRaceFault_1 (carga 270, a de menor força média) foram classificadas como normais, e 6 janelas normais (2 da baseline_1 e 4 da baseline_3) como pista externa. Outras 8 janelas da OuterRaceFault_vload_7 (carga 300) foram classificadas como pista interna. Com as medidas do ritmo, houve 3 erros em 312 janelas: uma janela da gravação de pista interna com carga 200 classificada como normal, a mesma gravação fraca da detecção, e uma janela da baseline_2 e outra da baseline_3 classificadas como pista interna.

O teste com PCA, pela validação separada por gravação:

| Entrada | Regressão logística | Knn (k=3) | Árvore de decisão | Random Forest |
|---|---|---|---|---|
| 6 medidas, sem PCA | 0,972 | 0,992 | 1,000 | 0,992 |
| PCA, 70% da variação (1 componente) | 0,767 | 0,821 | 0,813 | 0,813 |
| PCA, 90% da variação (2 componentes) | 0,975 | 0,994 | 0,994 | 0,997 |

Com o corte de 70% da variação, o PCA piorou o diagnóstico: o primeiro componente sozinho já explica 77,6% da variação, então só ele fica, e o acerto cai para 76,7% a 82,1%. O PCA guarda as direções em que os dados mais variam, mas não usa as classes. A direção que mais varia aqui é a força geral da vibração, que é justamente o que confunde o normal com a pista externa. Com 90% da variação ficam 2 componentes, e o resultado volta a ser o mesmo das 6 medidas sem PCA. Com tão poucas medidas, comprimir não trouxe ganho. Juntar as 6 medidas deu resultado parecido com o de usar só as do ritmo (de 97,2% a 100%, contra 98,6% a 99,0%).

## 6. Tabela resumo

A tabela segue o formato pedido para o projeto, com uma linha por combinação de técnica e medidas. Na detecção, TPR e FPR são as médias das 3 rodadas. No diagnóstico, TPR é a fração das janelas com defeito que não foram classificadas como normais, FPR é a fração das janelas normais classificadas com algum defeito, e a acurácia por classe vem da avaliação por gravação. As matrizes de confusão completas estão na seção 5.

In [ ]:
# Tabela resumo no formato pedido: uma linha por combinação de técnica e medidas. Na detecção, TPR e
# FPR são as médias das 3 rodadas. No diagnóstico, TPR é a fração das janelas com defeito que não
# foram classificadas como normais, FPR a fração das normais classificadas com algum defeito, e a
# acurácia por classe vem da avaliação por gravação.
linhas = []
for (tecnica, nome_medidas), r in deteccao.iterrows():
    linhas.append({'Dataset': 'MFPT', 'Técnica': f'{tecnica} (detecção, medidas {nome_medidas})',
                   'Tipo de falha identificada': 'não se aplica', 'Taxa de detecção (TPR)': round(r.TPR, 3),
                   'Falso positivo (FPR)': round(r.FPR, 3), 'Acurácia por classe': 'não se aplica'})
for nome_medidas, (previsto, escolhidos) in finais.items():
    defeito = janelas.classe != 'normal'
    acerto = (previsto == janelas.classe).groupby(janelas.classe).mean()
    linhas.append({'Dataset': 'MFPT', 'Técnica': f'{" / ".join(dict.fromkeys(escolhidos))} (diagnóstico, medidas {nome_medidas})',
                   'Tipo de falha identificada': 'pista externa, pista interna',
                   'Taxa de detecção (TPR)': round((previsto[defeito] != 'normal').mean(), 3),
                   'Falso positivo (FPR)': round((previsto[~defeito] != 'normal').mean(), 3),
                   'Acurácia por classe': ' · '.join(f'{c} {acerto[c]:.3f}' for c in classes_ordem)})

tabela_resumo = pd.DataFrame(linhas)
tabela_resumo

### Resultados

| Dataset | Técnica | Tipo de falha identificada | Taxa de detecção (TPR) | Falso positivo (FPR) | Acurácia por classe |
|---|---|---|---|---|---|
| MFPT | Isolation Forest (detecção, medidas genéricas) | não se aplica | 0,889 | 0,014 | não se aplica |
| MFPT | Isolation Forest (detecção, medidas do ritmo do defeito) | não se aplica | 0,107 | 0,000 | não se aplica |
| MFPT | Distância com PCA (detecção, medidas genéricas) | não se aplica | 1,000 | 0,014 | não se aplica |
| MFPT | Distância com PCA (detecção, medidas do ritmo do defeito) | não se aplica | 0,990 | 0,028 | não se aplica |
| MFPT | Regressão logística / Random Forest (diagnóstico, medidas genéricas) | pista externa, pista interna | 0,942 | 0,083 | normal 0,917 · pista externa 0,859 · pista interna 1,000 |
| MFPT | Knn (k=3) (diagnóstico, medidas do ritmo do defeito) | pista externa, pista interna | 0,996 | 0,028 | normal 0,972 · pista externa 1,000 · pista interna 0,988 |

## 7. Discussão

Os resultados indicam que as medidas baseadas no ritmo característico de cada defeito constituem o elemento mais robusto da abordagem. Calculado a partir da geometria do rolamento e da velocidade do eixo, esse ritmo aparece de 422,8 a 2.380,5 vezes acima do fundo nas gravações com defeito e não se destaca nas gravações normais (no máximo 4,1 vezes). Como medida de entrada, ele produziu um detector que, avaliado em gravações normais não vistas, detectou 99,0% das janelas com defeito com 2,8% de alarme falso (distância com PCA), e o diagnóstico mais estável em gravações não vistas (99,0% de acerto, contra 91,0% das medidas genéricas). Por depender da física do rolamento, e não das condições de cada sessão de gravação, essa medida é também a menos sujeita às diferenças de montagem entre as gravações normais e as de pista interna.

Na detecção, o desempenho dependeu mais das medidas fornecidas ao modelo do que da técnica empregada. A distância com PCA apresentou bom desempenho com os dois grupos de medidas. O Isolation Forest detectou apenas 10,7% dos defeitos quando alimentado com as medidas do ritmo. Esse resultado é coerente com o funcionamento do método: os cortes aleatórios são feitos entre o menor e o maior valor observado no treino, de modo que pontos muito acima dessa faixa recebem pontuação semelhante à da janela normal mais extrema, que define o limite de alerta. Com as medidas genéricas, o Isolation Forest detectou 88,9% dos defeitos, mas apenas 63,0% das janelas de pista externa com carga 270, o caso em que defeito e normal foram gravados nas mesmas condições. A técnica mais elaborada não implicou, portanto, melhor desempenho neste problema.

No diagnóstico, as medidas genéricas (força média, pico, curtose e pico dividido pela força média) chegaram a 95,8% a 100% de acerto na validação cruzada comum, mas a 89,2% a 91,4% na validação separada por gravação, e a 91,0% na avaliação final por gravação. A diferença indica que essas medidas carregam informação própria de cada gravação, que o modelo aproveita quando janelas da mesma gravação estão no treino e na avaliação. Os erros restantes se concentram entre o normal e a pista externa com carga 270 (80,6% de acerto nesse grupo). Com as medidas do ritmo, as duas validações coincidem e a avaliação final erra 3 de 312 janelas. A compressão das medidas por PCA, com o critério de 70% da variação, reduziu o acerto para 76,7% a 82,1%: o componente retido representa a intensidade geral da vibração, que é justamente a direção em que normal e pista externa se sobrepõem.

### Limitações

- O conjunto de dados contém apenas 3 gravações normais (18 s de sinal, 72 janelas), todas com carga de 270. O modelo do comportamento normal é estimado a partir de pouco material e de uma única condição de operação, e cada rodada da detecção mede o alarme falso numa única gravação de 24 janelas.
- As gravações de pista interna nunca foram feitas nas mesmas condições das normais. A única comparação em condições equivalentes é entre o rolamento normal e as três gravações de pista externa com carga de 270.
- As diferenças de taxa de gravação e de deslocamento do sensor entre sessões foram corrigidas, mas outras diferenças de sessão não registradas nos arquivos podem persistir.
- A faixa de 2 a 4 mil ciclos por segundo usada para extrair o ritmo foi tomada do script da MFPT, sem otimização nos nossos dados. A escolha dessa faixa pode mudar bastante a sensibilidade da análise da envoltória (BECHHOEFER; MENON, 2009).
- A documentação original do conjunto de dados não está mais disponível. As informações sobre o experimento foram obtidas dos próprios arquivos e dos scripts da MFPT distribuídos com os dados.

### Trabalhos futuros

- Incluir gravações normais em diferentes cargas, para separar o efeito da carga do efeito do defeito.
- Comparar outras faixas de frequência para a envoltória e escolher a faixa com dados de treino, como sugerem Bechhoefer e Menon (2009).
- Avaliar o efeito do tamanho da janela, já que janelas mais longas tornam o ritmo do defeito mais nítido, mas reduzem o número de exemplos normais para treino.
- Aplicar o detector às três gravações de máquinas reais que acompanham o conjunto de dados, cujos defeitos e frequências características são conhecidos.

## 8. Referências e uso de IA

**Dados**
- SOCIETY FOR MACHINERY FAILURE PREVENTION TECHNOLOGY (MFPT). Condition Based Maintenance Fault Database: bearing data. Página original (https://www.mfpt.org/fault-data-sets/) fora do ar em 2026. Cópia usada: https://www.kaggle.com/datasets/emperorpein/mfpt-fault-datasets.

**Ritmo do defeito (análise da envoltória)**
- BECHHOEFER, E.; MENON, P. Bearing Envelope Analysis Window Selection. In: Annual Conference of the Prognostics and Health Management Society, 2009.
- KIM, S.; AN, D.; CHOI, J.-H. Diagnostics 101: A Tutorial for Fault Diagnostics of Rolling Element Bearing Using Envelope Analysis in MATLAB. Applied Sciences, v. 10, n. 20, 7302, 2020. DOI: 10.3390/app10207302.

**Isolation Forest**
- LIU, F. T.; TING, K. M.; ZHOU, Z.-H. Isolation Forest. In: IEEE International Conference on Data Mining (ICDM), 2008, Pisa. p. 413-422.
- LIU, F. T.; TING, K. M.; ZHOU, Z.-H. Isolation-based anomaly detection. ACM Transactions on Knowledge Discovery from Data, v. 6, n. 1, p. 3:1-3:39, 2012.

**Distância com PCA**
- JOLLIFFE, I. T.; CADIMA, J. Principal component analysis: a review and recent developments. Philosophical Transactions of the Royal Society A, v. 374, n. 2065, 20150202, 2016. DOI: 10.1098/rsta.2015.0202.
- PÉREZ-TORRES, A. et al. Early fault detection in gearboxes via dynamic principal component analysis–driven multivariate statistical process control. PLOS One, v. 21, n. 5, e0348497, 2026. DOI: 10.1371/journal.pone.0348497.

**Classificadores (regressão logística, Knn, árvore de decisão, Random Forest)**
- SPERANDEI, S. Understanding logistic regression analysis. Biochemia Medica, v. 24, n. 1, p. 12-18, 2014. DOI: 10.11613/BM.2014.003.
- COVER, T. M.; HART, P. E. Nearest neighbor pattern classification. IEEE Transactions on Information Theory, v. IT-13, n. 1, p. 21-27, 1967. DOI: 10.1109/TIT.1967.1053964.
- QUINLAN, J. R. Induction of decision trees. Machine Learning, v. 1, n. 1, p. 81-106, 1986. DOI: 10.1023/A:1022643204877.
- BREIMAN, L. Random forests. Machine Learning, v. 45, n. 1, p. 5-32, 2001. DOI: 10.1023/A:1010933404324.
- PEDREGOSA, F. et al. Scikit-learn: Machine Learning in Python. Journal of Machine Learning Research, v. 12, p. 2825-2830, 2011.

**Validação separada por gravação**
- ROBERTS, D. R. et al. Cross-validation strategies for data with temporal, spatial, hierarchical, or phylogenetic structure. Ecography, v. 40, n. 8, p. 913-929, 2017. DOI: 10.1111/ecog.02881.
- KAPOOR, S.; NARAYANAN, A. Leakage and the reproducibility crisis in machine-learning-based science. Patterns, v. 4, n. 9, 100804, 2023. DOI: 10.1016/j.patter.2023.100804.

**Uso de IA**

Usamos o Claude (Anthropic), pelo Claude Code, como assistente durante o trabalho: para escrever e revisar o código, rodar e conferir as análises, levantar referências e redigir a primeira versão dos textos. As decisões sobre o método, a revisão dos resultados e o texto final são do grupo, que se responsabiliza pelo conteúdo.